# Laplacian-of-Gaussian blob detector

About 15 minutes. Detect bright blobs at several scales, check a synthetic blob, then find the apples.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tavisualcomputing/viscomp2026/blob/main/Exercises/W5/W5_log_exercise.ipynb)

## Before you start

Run the setup cell first. It uses the photos next to this notebook, and downloads the ones it needs from the course repository when they are missing (this is what happens in Colab). If the download fails, upload the `.jpg` files from `Exercises/W5` into the Colab file browser.


In [ ]:
%matplotlib inline
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage

plt.rcParams["figure.dpi"] = 80


IMAGE_BASE_URL = "https://raw.githubusercontent.com/tavisualcomputing/viscomp2026/main/Exercises/W5"


def find_image_dir():
    """Folder with the photos: next to the notebook, in images/, or in Exercises/W5."""
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        for folder in (cand, cand / "images", cand / "Exercises" / "W5"):
            if (folder / "line.jpg").is_file():
                return folder
    return start


IMAGE_DIR = find_image_dir()


def load_rgb(name):
    """Load a course photo. Downloads it on first use when it is not on disk."""
    path = IMAGE_DIR / name
    if not path.is_file():
        urllib.request.urlretrieve(f"{IMAGE_BASE_URL}/{name}", path)
    return np.asarray(Image.open(path).convert("RGB"))


def downscale(rgb, long_side=960):
    """Shrink so the long side is `long_side`. The JPEG on disk stays full HD."""
    image = Image.fromarray(rgb)
    image.thumbnail((long_side, long_side), Image.Resampling.LANCZOS)
    return np.asarray(image)


def rgb_to_gray(rgb):
    image = rgb.astype(np.float64)
    return (0.2989 * image[:, :, 0] + 0.5870 * image[:, :, 1] + 0.1140 * image[:, :, 2]) / 255.0


def image_gradients(gray):
    """Sobel derivatives. Ix is horizontal, Iy is vertical."""
    Ix = ndimage.sobel(gray, axis=1)
    Iy = ndimage.sobel(gray, axis=0)
    return Ix, Iy


In [ ]:
def red_minus_green(rgb):
    """Channel in which a red apple is a bright blob on green leaves."""
    image = rgb.astype(np.float64) / 255.0
    return image[:, :, 0] - image[:, :, 1]


def drop_border(blobs, shape, margin=8):
    height, width = shape
    return [
        blob for blob in blobs
        if margin <= blob[0] < width - margin and margin <= blob[1] < height - margin
    ]


def suppress_overlaps(blobs, min_separation=0.7):
    """Keep the stronger blob when two circles land on the same object."""
    ordered = sorted(blobs, key=lambda blob: -blob[3])
    kept = []
    for x, y, sigma, score in ordered:
        radius = np.sqrt(2.0) * sigma
        clear = True
        for x2, y2, sigma2, _ in kept:
            radius2 = np.sqrt(2.0) * sigma2
            limit = min_separation * max(radius, radius2)
            if (x - x2) ** 2 + (y - y2) ** 2 < limit ** 2:
                clear = False
                break
        if clear:
            kept.append((x, y, sigma, score))
    return kept


def draw_blobs(ax, blobs, color="crimson"):
    for x, y, sigma, _score in blobs:
        circle = plt.Circle(
            (x, y), np.sqrt(2.0) * sigma, fill=False, color=color, linewidth=0.9
        )
        ax.add_patch(circle)


## The detector

A blob of size $\sigma$ is a peak of the scale-normalized Laplacian of Gaussian:

$$
L(x, y, \sigma) = \sigma^2 \, \nabla^2 (G_\sigma * I)
$$

The factor $\sigma^2$ puts every scale on the same footing. Without it, the smallest filters produce the largest raw numbers and the detector always prefers tiny blobs.

**Bright or dark?** Look at a bright blob along one line: the smoothed intensity rises to a peak and falls again, like a hill. At the top of a hill the curve bends downward, so its second derivative is **negative**. The Laplacian is the sum of the second derivatives in $x$ and $y$, so at the center of a **bright** blob $L$ is **negative** (a minimum of $L$). A **dark** blob is a valley: the curve bends upward and $L$ is **positive**.

Our code looks for *maxima* of a score. To find bright blobs we therefore use the score $-L$, which turns the negative dip into a peak. To find dark blobs we use $+L$. The `polarity` argument of `blob_detections` (`"bright"` or `"dark"`) only chooses this sign. Everything after it is identical.

Apples and light balloons are bright blobs. The circle we draw has radius $\sigma\sqrt{2}$, which is where the LoG of a Gaussian blob crosses zero.

Stack $L$ over a list of $\sigma$ and keep pixels that are extrema in both space and scale.


## Your turn

The apple figure and the blob check both call these two functions.


In [ ]:
def scale_normalized_log(image, sigma):
    """Return sigma**2 * Laplacian(Gaussian_sigma * image).

    Blur with `ndimage.gaussian_filter`, then `ndimage.laplace`.
    """
    raise NotImplementedError


### Finding peaks in space and scale

The `blob_detections` stub gives you `local_max = ndimage.maximum_filter(volume, size=(3, 3, 3))`. If this is new to you, the Harris notebook explains it in detail. In short: it replaces every value by the **largest value in the block around it**. Here the block is 3 scales x 3 rows x 3 columns, so a voxel with `volume == local_max` is at least as large as all 26 neighbours across space and scale.

`np.nonzero` on a 3-D boolean mask returns three index arrays, one per axis: `scale_ids, ys, xs = np.nonzero(mask)`.


In [ ]:
def blob_detections(image, sigmas, rel_threshold=0.15, polarity="bright"):
    """Return a list of (x, y, sigma, score).

    polarity="bright": score = -(scale-normalized LoG). Apples use this.
    polarity="dark": score = +(scale-normalized LoG).
    Keep local maxima of the score in a 3x3x3 neighborhood across space
    and scale. Drop the first and last scale. Keep scores >=
    rel_threshold * the best score.
    """
    if polarity == "bright":
        sign = -1.0
    elif polarity == "dark":
        sign = 1.0
    else:
        raise ValueError("polarity must be 'bright' or 'dark'")

    # GIVEN: volume[i, y, x] is the score at scale sigmas[i].
    volume = np.stack(
        [sign * scale_normalized_log(image, float(sigma)) for sigma in sigmas]
    )
    # GIVEN: local_max[i, y, x] is the largest score in the 3x3x3 block
    # around (i, y, x): 3 scales x 3 rows x 3 columns.
    local_max = ndimage.maximum_filter(volume, size=(3, 3, 3))

    # Your code here:
    # 1. mask = True where volume equals local_max (the voxel wins its block).
    # 2. Set mask[0] and mask[-1] to False (no neighbor on one side).
    # 3. If the best score is <= 0 return []. Otherwise also require
    #    volume >= rel_threshold * best score.
    # 4. scale_ids, ys, xs = np.nonzero(mask)  (a 3-D mask gives three index
    #    arrays, in the order of the axes: scale, row, column).
    # 5. Return one (x, y, sigma, score) tuple per detection, with
    #    sigma = sigmas[scale_id] and score = volume[scale_id, y, x].
    raise NotImplementedError


## Check: one Gaussian blob

The image is a bright Gaussian with standard deviation 8, centered at $(x, y) = (50, 60)$. The strongest detection should sit on that center, at $\sigma = 8$.


In [ ]:
yy, xx = np.mgrid[0:120, 0:120]
blob = np.exp(-((yy - 60) ** 2 + (xx - 50) ** 2) / (2 * 8.0 ** 2))
sigmas = np.array([3.0, 5.0, 8.0, 12.0, 18.0])
found = blob_detections(blob, sigmas, rel_threshold=0.3, polarity="bright")
best = max(found, key=lambda item: item[3])
print("Strongest blob (x, y, sigma, score):", tuple(round(v, 3) for v in best))
assert abs(best[0] - 50) <= 1 and abs(best[1] - 60) <= 1
assert best[2] == 8.0
print("Blob check passed.")

# Visualization: the blob, the detection, and the score over scale at its center.
x0, y0, s0 = int(best[0]), int(best[1]), best[2]
scores = [-scale_normalized_log(blob, float(sg))[y0, x0] for sg in sigmas]

fig, axes = plt.subplots(1, 3, figsize=(9.5, 3.2))
axes[0].imshow(blob, cmap="gray")
axes[0].add_patch(plt.Circle((x0, y0), np.sqrt(2.0) * s0, fill=False, color="lime", linewidth=1.5))
axes[0].plot([x0], [y0], "r+")
axes[0].set_title(f"Gaussian blob (true sigma 8)\ndetected sigma {s0:g}, circle radius sigma*sqrt(2)")

response = scale_normalized_log(blob, float(s0))
limit = np.abs(response).max()
im = axes[1].imshow(response, cmap="seismic", vmin=-limit, vmax=limit)
axes[1].set_title(f"Scale-normalized LoG at sigma {s0:g}\n(bright blob = negative, blue)")
fig.colorbar(im, ax=axes[1], fraction=0.046)

axes[2].plot(sigmas, scores, "o-")
axes[2].axvline(8.0, color="gray", linestyle="--", label="true sigma = 8")
axes[2].set_xlabel("sigma")
axes[2].set_ylabel("score = -L at the blob center")
axes[2].set_title("Score peaks at the blob's own scale")
axes[2].legend()
for ax in axes[:2]:
    ax.set_axis_off()
fig.tight_layout()
plt.show()


## Apples

`images/blob2.jpg`. An apple is red on green leaves, so it is a bright blob in the red-minus-green image (that conversion is given). Detection runs on a 960-pixel-long copy; the file on disk is full HD.


In [ ]:
rgb = downscale(load_rgb("blob2.jpg"), long_side=960)
channel = red_minus_green(rgb)
sigmas = np.geomspace(3.0, 14.0, 8)
blobs = blob_detections(channel, sigmas, rel_threshold=0.18, polarity="bright")
blobs = suppress_overlaps(drop_border(blobs, channel.shape))
print(f"{len(blobs)} apple-scale blobs")

fig, ax = plt.subplots(figsize=(6, 7.5))
ax.imshow(rgb)
draw_blobs(ax, blobs, color="gold")
ax.set_title("LoG blobs on the apple tree")
ax.set_axis_off()
fig.tight_layout()
plt.show()


## Balloons

`images/blob1.jpg`, in grayscale. This is a harder scene and a good place to see what LoG does and does not do. It finds blobs of any kind, not balloons: the rocky terrain is full of blob-shaped bumps, shadows, and tree clumps, so many circles miss. The balloons that are lighter than the ground are bright blobs. The two in the sky are darker than the sky, so this cell asks for both polarities, but only the 40 strongest are drawn and those two are weak. Drawing them would need a lower threshold.


In [ ]:
rgb = downscale(load_rgb("blob1.jpg"), long_side=960)
gray = rgb_to_gray(rgb)
sigmas = np.geomspace(5.0, 24.0, 8)
blobs = blob_detections(gray, sigmas, rel_threshold=0.3, polarity="bright")
blobs += blob_detections(gray, sigmas, rel_threshold=0.3, polarity="dark")
blobs = suppress_overlaps(drop_border(blobs, gray.shape))
blobs = sorted(blobs, key=lambda item: -item[3])[:40]
print(f"drawing {len(blobs)} balloon-scale blobs")

fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(rgb)
draw_blobs(ax, blobs)
ax.set_title("LoG blobs on the balloons")
ax.set_axis_off()
fig.tight_layout()
plt.show()


## Look at the figures

1. In the apple figure, the circles should sit on apples and their radii should be about the apple radii. A circle on a leaf is a blob in the red-minus-green image as well.
2. Delete the `sigma ** 2` factor inside `scale_normalized_log` and rerun the apple cell. The figure comes back empty: without that factor the smallest filter always wins the comparison across scale, and the smallest scale is discarded because it has no finer neighbor.
3. On the balloons, which polarity can fire on a balloon that is darker than the sky behind it?
4. LoG alone does not know what a balloon is. Name one cue (color, texture, size) you would add to separate balloons from rocks.
